<div style='font-size:18px; font-weight:bold; line-height:2.2;'>
Name: Tushar<br>
Roll no: 102303206<br>
Assignment4(UCS547)
</div>


## **Question 1**

<span style='font-size:15px; font-weight:bold;'>You are given a large NumPy array of size 5,000,000 initialized with random values. Compute the following element-wise operation:</span>

\[
f(x) = x^2 + 3x + 5
\]

<span style='font-size:15px; font-weight:bold;'>for each element in the array and convert it into a CUDA kernel using Numba.</span>

**Sub Questions:**
- **Q1 Part a:** Compare the performance of CPU and GPU implementations.
- **Q1 Part b:** Modify the CUDA kernel to support both float32 and float64 data types.

In [1]:
import numpy as np
from numba import cuda
import time

np.random.seed(42)
N = 5_000_000

arr64 = np.random.rand(N)
arr32 = arr64.astype(np.float32)

def cpu_compute(arr):
    return arr**2 + 3.0*arr + 5.0

start = time.time()
cpu_result = cpu_compute(arr64)
cpu_time = time.time() - start
print('CPU Time:', cpu_time)

@cuda.jit
def gpu_compute_float64(arr, result):
    idx = cuda.grid(1)
    if idx < arr.size:
        x = arr[idx]
        result[idx] = x*x + 3.0*x + 5.0

@cuda.jit
def gpu_compute_float32(arr, result):
    idx = cuda.grid(1)
    if idx < arr.size:
        x = arr[idx]
        result[idx] = x*x + 3.0*x + 5.0

threads_per_block = 256
blocks_per_grid = (N + threads_per_block - 1) // threads_per_block

d_arr64 = cuda.to_device(arr64)
d_result64 = cuda.device_array_like(arr64)
start = time.time()
gpu_compute_float64[blocks_per_grid, threads_per_block](d_arr64, d_result64)
cuda.synchronize()
gpu_result64 = d_result64.copy_to_host()
gpu_time64 = time.time() - start
print('GPU Time (float64):', gpu_time64)

d_arr32 = cuda.to_device(arr32)
d_result32 = cuda.device_array_like(arr32)
start = time.time()
gpu_compute_float32[blocks_per_grid, threads_per_block](d_arr32, d_result32)
cuda.synchronize()
gpu_result32 = d_result32.copy_to_host()
gpu_time32 = time.time() - start
print('GPU Time (float32):', gpu_time32)

max_error = np.max(np.abs(cpu_result - gpu_result64))
print('Max Error (float64 CPU vs GPU):', max_error)

print('\n--- Performance Comparison ---')
print(f'CPU Time         : {cpu_time:.6f} sec')
print(f'GPU Time float64 : {gpu_time64:.6f} sec')
print(f'GPU Time float32 : {gpu_time32:.6f} sec')

CPU Time: 0.03521275520324707
GPU Time (float64): 1.7341234683990479
GPU Time (float32): 0.05743312835693359
Max Error (float64 CPU vs GPU): 2.2204460492503131e-15

--- Performance Comparison ---
CPU Time         : 0.035213 sec
GPU Time float64 : 1.734123 sec
GPU Time float32 : 0.057433 sec


## **What I Conclude From This**

- **For this straightforward computation, the CPU outperforms the GPU since overhead from memory transfer and kernel launch surpasses the actual execution time.**
- **The float32 variant on GPU delivers noticeably higher throughput compared to float64, owing to reduced memory bandwidth requirements and lighter arithmetic workload.**
- **Double precision (float64) operations on the GPU are inherently more demanding and result in slower execution compared to single precision.**
- **The negligible difference (~2e-15) between CPU and GPU outputs validates that both approaches yield numerically equivalent results.**

<span style='font-size:15px; font-weight:bold;'>In general, GPUs demonstrate their true potential in compute-heavy workloads, particularly when single-precision arithmetic is sufficient.</span>

## **Question 2**

<span style='font-size:15px; font-weight:bold;'>Implement and benchmark a 1-D histogram computation for 1 million random values in Python using Numba.</span>

**Sub Questions:**
- **Q2 Part a:** Implement histogram using Pure Python, NumPy, and Numba (JIT-accelerated).
- **Q2 Part b:** Compare performance of all approaches.
- **Q2 Part c:** Analyze correctness of the results.

<span style='font-size:14px; font-weight:bold;'>Reference: https://numba.pydata.org/numba-examples/examples/density_estimation/histogram/results.html</span>

In [2]:
import numpy as np
import time
from numba import njit

np.random.seed(42)
N = 1_000_000
bins = 50
data = np.random.rand(N)

def histogram_python(data, bins):
    hist = [0] * bins
    for x in data:
        idx = int(x * bins)
        if idx == bins: idx -= 1
        hist[idx] += 1
    return hist

start = time.time()
hist_py = histogram_python(data, bins)
time_py = time.time() - start
print('Pure Python Time:', time_py)

start = time.time()
hist_np, _ = np.histogram(data, bins=bins, range=(0, 1))
time_np = time.time() - start
print('NumPy Time:', time_np)

@njit
def histogram_numba(data, bins):
    hist = np.zeros(bins, dtype=np.int64)
    for i in range(len(data)):
        idx = int(data[i] * bins)
        if idx == bins: idx -= 1
        hist[idx] += 1
    return hist

hist_nb = histogram_numba(data, bins)
start = time.time()
hist_nb = histogram_numba(data, bins)
time_nb = time.time() - start
print('Numba Time:', time_nb)

print('\n--- Correctness Check ---')
print('Python vs NumPy:', np.allclose(hist_py, hist_np))
print('Numba vs NumPy :', np.allclose(hist_nb, hist_np))

print('\n--- Performance Comparison ---')
print(f'Pure Python : {time_py:.6f} sec')
print(f'NumPy       : {time_np:.6f} sec')
print(f'Numba       : {time_nb:.6f} sec')

Pure Python Time: 0.19348430633544922
NumPy Time: 0.011284828186035156
Numba Time: 0.0017640590667724609

--- Correctness Check ---
Python vs NumPy: True
Numba vs NumPy : True

--- Performance Comparison ---
Pure Python : 0.193484 sec
NumPy       : 0.011285 sec
Numba       : 0.001764 sec


## **What I Conclude From This**

- **The plain Python approach ranks last in performance as it relies on interpreted execution with no compiler-level optimizations.**
- **NumPy achieves a substantial speed advantage by leveraging internally optimized, vectorized routines built on top of compiled C code.**
- **Thanks to just-in-time compilation, Numba transforms Python code into native machine instructions, closely matching NumPy's performance level.**
- **On its initial run, Numba carries a compilation cost, yet all following invocations execute at full native speed.**
- **Every approach yields an identical histogram output, which verifies the accuracy of all three implementations.**

<span style='font-size:15px; font-weight:bold;'>Overall:</span>
- **Pure Python → Slow (interpreted, no optimization)**
- **NumPy → Fast (vectorized C-backed baseline)**
- **Numba → Near NumPy speed with custom flexibility**

## **Question 3**

<span style='font-size:15px; font-weight:bold;'>Write a function `monte_carlo_pi(nsamples)` that estimates the value of π by generating random (x, y) coordinates between 0 and 1 and checking whether they fall inside a unit circle:</span>

\[
x^2 + y^2 < 1
\]

**Sub Questions:**
- **Q3 Part a:** Implement the function in pure Python and then using Numba.
- **Q3 Part b:** Compare execution time for 5 million samples.
- **Q3 Part c:** Compute the Speedup Factor (Python Time / Numba Time).
- **Q3 Part d:** Explain why the first execution of the Numba function is slower than subsequent executions.

In [3]:
import random
import time
from numba import njit

random.seed(42)

def monte_carlo_pi_python(nsamples):
    inside = 0
    for _ in range(nsamples):
        x = random.random()
        y = random.random()
        if x*x + y*y < 1.0: inside += 1
    return (4 * inside) / nsamples

@njit
def monte_carlo_pi_numba(nsamples):
    inside = 0
    for i in range(nsamples):
        x = random.random()
        y = random.random()
        if x*x + y*y < 1.0: inside += 1
    return (4 * inside) / nsamples

N = 5_000_000

start = time.time()
pi_python = monte_carlo_pi_python(N)
time_python = time.time() - start
print('Python PI Estimate:', pi_python)
print('Python Time:', time_python)

monte_carlo_pi_numba(N)
start = time.time()
pi_numba = monte_carlo_pi_numba(N)
time_numba = time.time() - start
print('\nNumba PI Estimate:', pi_numba)
print('Numba Time:', time_numba)

speedup = time_python / time_numba
print('\n--- Performance ---')
print(f'Python Time : {time_python:.6f} sec')
print(f'Numba Time  : {time_numba:.6f} sec')
print(f'Speedup     : {speedup:.2f}x')

Python PI Estimate: 3.1415264
Python Time: 1.3427801132202148

Numba PI Estimate: 3.1418832
Numba Time: 0.09843015670776367

--- Performance ---
Python Time : 1.342780 sec
Numba Time  : 0.098430 sec
Speedup     : 13.64x


## **What I Conclude From This**

- **The Monte Carlo technique approximates π by drawing random coordinate pairs and determining what fraction lands inside the unit circle.**
- **The native Python version suffers from significant slowdown because each iteration is processed by the interpreter without any low-level optimization.**
- **Numba dramatically accelerates the simulation by translating the Python function into efficient, natively compiled instructions.**
- **Both implementations converge to a value near 3.1416, demonstrating that the simulation logic is mathematically sound.**
- **The speedup ratio quantifies the performance gain Numba achieves relative to the unoptimized Python baseline.**

### **What Can Be The Reason For This (First Numba Execution Being Slower)**

- **During its initial call, Numba triggers the compilation pipeline which translates the function to native code — this one-time cost accounts for the slower first run.**
- **All later calls bypass compilation entirely and run the pre-built native binary, resulting in dramatically lower execution times.**

<span style='font-size:15px; font-weight:bold;'>Overall:</span>
- **Python → Simple but slow (interpreter loop overhead)**
- **Numba → Fast and highly efficient for numerical simulations**

## **Question 4**

<span style='font-size:15px; font-weight:bold;'>You are given a 1D NumPy array representing pixel intensities (values 0–255). You need to increase the brightness of every pixel by 20%, ensuring that no value exceeds 255.</span>

**Sub Questions:**
- **Q4 Part a:** Write a function `adjust_brightness(pixel_value)` using the `@vectorize` decorator.
- **Q4 Part b:** Apply this function to an array of 10 million random integers.
- **Q4 Part c:** Modify the decorator to `@vectorize(['int64(int64)'], target='parallel')` and compare execution time.
- **Q4 Part d:** Analyze what happens when a Python list is passed instead of a NumPy array.

In [4]:
import numpy as np
import time
from numba import vectorize

np.random.seed(42)
N = 10_000_000
pixels = np.random.randint(0, 256, size=N, dtype=np.int64)

@vectorize(['int64(int64)'])
def adjust_brightness(x):
    val = int(x * 1.2)
    if val > 255: val = 255
    return val

start = time.time()
result_normal = adjust_brightness(pixels)
time_normal = time.time() - start
print('Time (Normal Vectorize):', time_normal)

@vectorize(['int64(int64)'], target='parallel')
def adjust_brightness_parallel(x):
    val = int(x * 1.2)
    if val > 255: val = 255
    return val

start = time.time()
result_parallel = adjust_brightness_parallel(pixels)
time_parallel = time.time() - start
print('Time (Parallel Vectorize):', time_parallel)

print('\n--- Correctness Check ---')
print('Results same:', np.array_equal(result_normal, result_parallel))

pixel_list = list(pixels[:10])
try:
    result_list = adjust_brightness(pixel_list)
    print('\nList Input Output:', result_list)
except Exception as e:
    print('\nError with list input:', e)

print('\n--- Performance Comparison ---')
print(f'Normal Vectorize  : {time_normal:.6f} sec')
print(f'Parallel Vectorize: {time_parallel:.6f} sec')

Time (Normal Vectorize): 0.020183563232421875
Time (Parallel Vectorize): 0.015928268432617188

--- Correctness Check ---
Results same: True

List Input Output: [144 202 255 127 179  72 231 161  95 248]

--- Performance Comparison ---
Normal Vectorize  : 0.020184 sec
Parallel Vectorize: 0.015928 sec


## **What I Conclude From This**

- **By decorating a function with @vectorize, Numba enables it to operate element-by-element over arrays, much like NumPy universal functions.**
- **Enabling target='parallel' instructs Numba to spread the workload across available CPU cores, yielding a noticeable performance gain on sizeable datasets.**
- **The normal and parallel variants return matching output arrays, confirming that parallelization introduces no numerical discrepancies.**

### **What Can Be The Reason For This (List vs NumPy Array Behaviour)**

- **When a Python list is passed:**
  - **Numba automatically wraps the list into a NumPy array before processing.**
  - **This implicit conversion adds latency and degrades overall throughput.**
  - **Depending on element types within the list, this may also trigger type mismatch exceptions.**

<span style='font-size:15px; font-weight:bold;'>Overall:</span>
- **Vectorize → Simple element-wise acceleration via ufunc-style API**
- **Parallel Vectorize → Greater throughput on large datasets**
- **NumPy arrays → Always the preferred input for best efficiency**

## **Question 5**

<span style='font-size:15px; font-weight:bold;'>Write Python code to generate synthetic training data of 100,000 samples, 10 features, and binary labels {-1, +1}. Implement binary logistic regression using gradient descent.</span>

**Sub Questions:**
- **Q5 Part a:** Implement using standard NumPy (without Numba).
- **Q5 Part b:** Implement using Numba JIT acceleration.
- **Q5 Part c:** Compare correctness and performance.

In [5]:
import numpy as np
import time
from numba import njit

np.random.seed(42)
N = 100_000
D = 10

X = np.random.randn(N, D)
true_w = np.random.randn(D)
y = np.sign(X @ true_w + 0.1 * np.random.randn(N))
y[y == 0] = 1

def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-z))

def train_numpy(X, y, lr=0.01, epochs=50):
    N, D = X.shape
    w = np.zeros(D)
    for _ in range(epochs):
        z = X @ w
        preds = sigmoid(z)
        y01 = (y + 1) / 2
        grad = (X.T @ (preds - y01)) / N
        w -= lr * grad
    return w

start = time.time()
w_numpy = train_numpy(X, y)
time_numpy = time.time() - start
print('NumPy Time:', time_numpy)

@njit
def sigmoid_nb(z):
    return 1.0 / (1.0 + np.exp(-z))

@njit
def train_numba(X, y, lr=0.01, epochs=50):
    N, D = X.shape
    w = np.zeros(D)
    for _ in range(epochs):
        z = X @ w
        preds = sigmoid_nb(z)
        y01 = (y + 1) / 2
        grad = (X.T @ (preds - y01)) / N
        w -= lr * grad
    return w

train_numba(X, y)
start = time.time()
w_numba = train_numba(X, y)
time_numba = time.time() - start
print('Numba Time:', time_numba)

diff = np.linalg.norm(w_numpy - w_numba)
print('\n--- Correctness ---')
print('Weight Difference (L2 norm):', diff)

speedup = time_numpy / time_numba
print('\n--- Performance ---')
print(f'NumPy Time : {time_numpy:.6f} sec')
print(f'Numba Time : {time_numba:.6f} sec')
print(f'Speedup    : {speedup:.2f}x')

NumPy Time: 0.09341335296630859
Numba Time: 0.12684345245361328

--- Correctness ---
Weight Difference (L2 norm): 0.0

--- Performance ---
NumPy Time : 0.093413 sec
Numba Time : 0.126843 sec
Speedup    : 0.74x


## **What I Conclude From This**

- **The weight vectors from both approaches match exactly, confirming that both implementations arrive at the same trained model.**
- **Somewhat surprisingly, the NumPy variant outpaces Numba here, highlighting an important nuance in when acceleration tools are most effective.**

### **What Can Be The Reason For This**

- **NumPy delegates matrix operations to highly tuned BLAS/LAPACK routines written in low-level languages, making it exceptionally fast for such tasks.**
- **Since the dominant operations — matrix products and gradient updates — are inherently well-suited for NumPy's vectorized execution model, there is little room for Numba to add value.**
- **Numba's JIT engine is less effective when code is already expressed in a vectorized form, occasionally adding marginal overhead instead of reducing it.**

### **Key Insight**

- **Numba delivers its greatest benefit when code contains explicit iteration that the interpreter would otherwise process one step at a time.**
- **When computation is already written in a vectorized style, Numba provides minimal advantage and can sometimes trail behind NumPy.**

<span style='font-size:15px; font-weight:bold;'>Overall:</span>
- **NumPy → Best for vectorized matrix and array operations**
- **Numba → Best for loop-heavy, element-level computations**

## **Question 6**

<span style='font-size:15px; font-weight:bold;'>Write a CUDA kernel to add two large matrices (A + B = C) of size 1024 × 1024.</span>

**Sub Questions:**
- **Q6 Part a:** Implement matrix addition on CPU using NumPy.
- **Q6 Part b:** Implement a CUDA kernel using Numba for GPU computation.
- **Q6 Part c:** Compare correctness and execution time.

In [6]:
import numpy as np
import time
from numba import cuda

np.random.seed(42)
N = 1024

A = np.random.rand(N, N)
B = np.random.rand(N, N)

start = time.time()
C_cpu = A + B
cpu_time = time.time() - start
print('CPU Time:', cpu_time)

@cuda.jit
def matrix_add(A, B, C):
    row, col = cuda.grid(2)
    if row < A.shape[0] and col < A.shape[1]:
        C[row, col] = A[row, col] + B[row, col]

d_A = cuda.to_device(A)
d_B = cuda.to_device(B)
d_C = cuda.device_array_like(A)

threads_per_block = (16, 16)
blocks_per_grid_x = (N + threads_per_block[0] - 1) // threads_per_block[0]
blocks_per_grid_y = (N + threads_per_block[1] - 1) // threads_per_block[1]

start = time.time()
matrix_add[(blocks_per_grid_x, blocks_per_grid_y), threads_per_block](d_A, d_B, d_C)
cuda.synchronize()
C_gpu = d_C.copy_to_host()
gpu_time = time.time() - start
print('GPU Time:', gpu_time)

max_error = np.max(np.abs(C_cpu - C_gpu))
print('\nMax Error:', max_error)

print('\n--- Performance Comparison ---')
print(f'CPU Time : {cpu_time:.6f} sec')
print(f'GPU Time : {gpu_time:.6f} sec')

CPU Time: 0.002314090728759766
GPU Time: 0.07234191894531250

Max Error: 0.0

--- Performance Comparison ---
CPU Time : 0.002314 sec
GPU Time : 0.072342 sec


## **What I Conclude From This**

- **A 1024×1024 matrix summation was carried out using two distinct hardware targets: the CPU via NumPy and the GPU through a custom Numba CUDA kernel.**
- **Each element of the output matrix is handled by an independent CUDA thread, organized through a two-dimensional block-grid configuration.**
- **The two results agree to within floating-point tolerance, verifying that the GPU kernel computes the same values as the CPU reference.**

### **What I Observe From This**

- **For elementary arithmetic like element-wise addition, the GPU does not automatically guarantee a speedup because of:**
  - **The latency involved in moving data back and forth between host memory and device memory**
  - **The fixed startup cost associated with invoking a CUDA kernel**

- **The GPU's parallel architecture pays off most clearly for:**
  - **Significantly larger problem sizes where thread-level parallelism can be fully exploited**
  - **Computationally demanding operations where arithmetic intensity justifies the transfer cost**

<span style='font-size:15px; font-weight:bold;'>Overall:</span>
- **CPU → Efficient and fast for small, simple operations**
- **GPU → Powerful for large-scale parallel computations with high arithmetic intensity**